# Accuracy precision and recall

We predict whether a bank customer subscribed. We first look at accuracy,
then count the mistakes and calculate precision and recall.

All the Python is in this notebook. Keep `bank-additional-full.csv` beside it.
If needed, install the libraries once in a notebook cell with
`%pip install pandas scikit-learn`.

Data: UCI Bank Marketing, Moro, Rita & Cortez (2014),
[CC BY 4.0](https://creativecommons.org/licenses/by/4.0/),
[DOI 10.24432/C5K306](https://doi.org/10.24432/C5K306).

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score

## 1. Look at the data

`y` is the recorded outcome: `yes` means the customer subscribed.

In [3]:
data = pd.read_csv("data.csv", sep=";")
data[["age", "campaign", "previous", "y"]].head()

,age,campaign,previous,y
0,56,1,0,no
1,57,1,0,no
2,37,1,0,no
3,40,1,0,no
4,56,1,0,no


In [4]:
data["y"].value_counts().rename("Customers").to_frame()

,Customers
y,
no,36548
yes,4640


## 2. Train one small model

Use the numeric columns to keep the example short. Remove call `duration`:
it is only known after the call, and our imagined decision happens before it.

Train on 80% of the records and keep 20% aside for evaluation.
The decision tree has a maximum depth of four. The random seed fixes the example.

In [5]:
X = data.select_dtypes(include="number").drop(columns="duration")
y = data["y"].eq("yes").astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

model = DecisionTreeClassifier(max_depth=4, min_samples_leaf=20, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

## 3. Start with accuracy

Accuracy asks: **of all the test records, how many predictions were correct?**
Compare the model with a rule that predicts "no" for everyone.

In [6]:
accuracy = accuracy_score(y_test, y_pred)
always_no_accuracy = (y_test == 0).mean()

print(f"Model accuracy:     {accuracy:.1%}")
print(f"Always-no accuracy: {always_no_accuracy:.1%}")

Model accuracy:     90.1%
Always-no accuracy: 88.7%


## 4. Count the mistakes

Rows show the recorded outcome. Columns show the prediction.
The bottom-right cell contains subscribers the model found.
The top-right cell contains non-subscribers the model flagged.

In [7]:
tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=[0, 1]).ravel()

pd.DataFrame(
    [[tn, fp], [fn, tp]],
    index=["Actually no", "Actually yes"],
    columns=["Predicted no", "Predicted yes"],
)

,Predicted no,Predicted yes
Actually no,7209,101
Actually yes,711,217


## 5. Now calculate precision

Precision asks: **of the customers the model flagged, how many subscribed?**

`precision = true positives / (true positives + false positives)`

In [8]:
precision = precision_score(y_test, y_pred)

print(f"Flagged customers: {tp + fp:,}")
print(f"Subscribed:        {tp:,}")
print(f"Did not subscribe: {fp:,}")
print(f"Precision: {tp} / ({tp} + {fp}) = {precision:.1%}")

Flagged customers: 318
Subscribed:        217
Did not subscribe: 101
Precision: 217 / (217 + 101) = 68.2%


## 6. Recall and the subscribers the model missed

Recall asks: **of all the actual subscribers, how many did the model find?**

`recall = true positives / (true positives + false negatives)`

The model found 217 of 928 subscribers: **23.4% recall**.
It missed 711 subscribers: **76.6% of the actual positives**.

Compare the three questions below. Each metric uses a different denominator.

In [9]:
recall = tp / (tp + fn)
print(f"Recall: {tp} / ({tp} + {fn}) = {recall:.1%}")
print(f"Missed: {fn} of {tp + fn} subscribers ({fn / (tp + fn):.1%})")

pd.DataFrame({
    "Question": ["How often was the model correct overall?",
                 "How often was a positive prediction correct?",
                 "How many actual subscribers did the model find?"],
    "Result": [f"{accuracy:.1%}", f"{precision:.1%}", f"{recall:.1%}"],
}, index=["Accuracy", "Precision", "Recall"])

Recall: 217 / (217 + 711) = 23.4%
Missed: 711 of 928 subscribers (76.6%)


,Question,Result
Accuracy,How often was the model correct overall?,90.1%
Precision,How often was a positive prediction correct?,68.2%
Recall,How many actual subscribers did the model find?,23.4%


## What the score did not tell us

The model is 90.1% accurate overall. Among its 318 positive predictions,
217 are subscribers: precision is 68.2%. Its recall is only 23.4%: it finds 217 of 928 subscribers and misses 711 (76.6%).

Read the counts behind the score before using predictions for a decision.

This is a teaching example using a random holdout of historical contacted
customers. It does not establish the effect of a new calling policy or prove
that this model is ready to deploy.

Dataset source: https://archive.ics.uci.edu/dataset/222/bank+marketing
The bundled CSV is unchanged. The notebook selects columns and encodes the
target in memory; no modified dataset is redistributed.